# 02 — Data Cleaning
**Input**: `data/raw/Telco_customer_churn.xlsx`  
**Output**: `data/processed/telco_churn_cleaned.csv`

## 1. Imports & Load

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')

Loaded: 7043 rows × 33 columns


## 2. Define Columns to Drop

In [2]:
ZERO_VARIANCE    = ['Country', 'State', 'Count']
IDENTIFIERS      = ['CustomerID', 'Lat Long', 'City', 'Zip Code', 'Latitude', 'Longitude']
LEAKAGE          = ['Churn Score', 'CLTV']
REDUNDANT_TARGET = ['Churn Label']
HIGH_MISSING     = ['Churn Reason']

DROP_COLS = ZERO_VARIANCE + IDENTIFIERS + LEAKAGE + REDUNDANT_TARGET + HIGH_MISSING
print(f'Columns to drop ({len(DROP_COLS)}):')
for c in DROP_COLS:
    print(f'  - {c}')

Columns to drop (13):
  - Country
  - State
  - Count
  - CustomerID
  - Lat Long
  - City
  - Zip Code
  - Latitude
  - Longitude
  - Churn Score
  - CLTV
  - Churn Label
  - Churn Reason


## 3. Drop Columns

In [3]:
df = df.drop(columns=DROP_COLS, errors='ignore')
print(f'✅ Shape after dropping: {df.shape}')
print(f'Remaining columns: {list(df.columns)}')

✅ Shape after dropping: (7043, 20)
Remaining columns: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value']


## 4. Fix Data Types

In [4]:
print('Before:', df['Total Charges'].dtype)
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
print('After :', df['Total Charges'].dtype)
print('NaN introduced:', df['Total Charges'].isnull().sum())

Before: object
After : float64
NaN introduced: 11


## 5. Handle Missing Values

In [5]:
null_mask = df['Total Charges'].isnull()
print(f'Rows with NaN Total Charges: {null_mask.sum()}')
print(df[null_mask][['Tenure Months', 'Monthly Charges', 'Total Charges']].head())

# New customers (Tenure=0) — fill with Monthly Charges
df.loc[null_mask, 'Total Charges'] = df.loc[null_mask, 'Monthly Charges']
print(f'✅ Missing values remaining: {df.isnull().sum().sum()}')

Rows with NaN Total Charges: 11
      Tenure Months  Monthly Charges  Total Charges
2234              0            52.55            NaN
2438              0            20.25            NaN
2568              0            80.85            NaN
2667              0            25.75            NaN
2856              0            56.05            NaN
✅ Missing values remaining: 0


## 6. Remove Duplicates

In [6]:
before = len(df)
df = df.drop_duplicates()
print(f'✅ Duplicates removed: {before - len(df)}  ({before} → {len(df)} rows)')

✅ Duplicates removed: 22  (7043 → 7021 rows)


## 7. Validation

In [7]:
print('=' * 50)
print('CLEANING SUMMARY')
print('=' * 50)
print(f'Final shape    : {df.shape}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Duplicates     : {df.duplicated().sum()}')
print(f'\nTarget balance :')
print(df['Churn Value'].value_counts())
print(f'Churn rate     : {df["Churn Value"].mean()*100:.2f}%')
print(f'\nDtypes:')
print(df.dtypes)

CLEANING SUMMARY
Final shape    : (7021, 20)
Missing values : 0
Duplicates     : 0

Target balance :
Churn Value
0    5164
1    1857
Name: count, dtype: int64
Churn rate     : 26.45%

Dtypes:
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges        float64
Churn Value            int64
dtype: object


## 8. Save

In [8]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/telco_churn_cleaned.csv', index=False)
print('✅ Saved: data/processed/telco_churn_cleaned.csv')
print(f'Shape: {df.shape}')


✅ Saved: data/processed/telco_churn_cleaned.csv
Shape: (7021, 20)
